# Objective

A higher education analyst wants to understand the important factors that influence a college's graduation rate. The dataset contains academic, financial, demographic, and institutional characteristics of colleges. The objective of this exercise is to explore the dataset, understand relationships between variables, prepare the data for linear models, and build regularised MLR models for predicting graduation rate.

*Import necessary libraries before proceeding*

In [1]:
import numpy as np; import pandas as pd  # data processing and transformations
import matplotlib.pyplot as plt; import seaborn as sns  # visualisation libraries
from sklearn.preprocessing import MinMaxScaler  # scaling
from sklearn.model_selection import train_test_split  # splitting data into train and test
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV  # linear regression model
from sklearn.metrics import r2_score, mean_squared_error  # performance metrics
import warnings; warnings.filterwarnings('ignore')  # suppress warnings
np.random.seed(42)  # Global reproducibility for numpy

# Data Preparation and EDA

### Task 1

*Load the dataset, study data frame dimensions, check column names, data types, and missing values*

In [2]:
df = pd.read_csv('college_data_full_set.csv'); df.sample()

,College Name,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,...,Expend,Grad.Rate,Total_Cost,Acceptance_Rate,Faculty_Quality_Index,Student_Success_Index,Region,College_Size,Funding_Type,Academic_Standing
596,Trinity University,Yes,2425.0,1818,601,62.0,93,2110,95,9998.0,...,8415.0,93,18380,0.749691,88.4,65.759369,West,Medium,Private,Elite


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   College Name           782 non-null    object 
 1   Private                782 non-null    object 
 2   Apps                   782 non-null    float64
 3   Accept                 782 non-null    int64  
 4   Enroll                 782 non-null    int64  
 5   Top10perc              782 non-null    float64
 6   Top25perc              782 non-null    int64  
 7   F.Undergrad            782 non-null    int64  
 8   P.Undergrad            782 non-null    int64  
 9   Outstate               782 non-null    float64
 10  Room.Board             782 non-null    float64
 11  Books                  782 non-null    int64  
 12  Personal               782 non-null    int64  
 13  PhD                    782 non-null    float64
 14  Terminal               782 non-null    float64
 15  S.F.Ra

### Task 2

*Study descriptive summary statistics of the data*

In [4]:
df.describe(include = 'all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
College Name,782,777,University of Cincinnati,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Private,782,2,Yes,567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apps,782.0,NaN,NaN,NaN,2906.350384,3804.521019,-500.0,787.0,1549.0,3421.5,48094.0
Accept,782.0,NaN,NaN,NaN,2019.932225,2448.107744,72.0,601.75,1110.0,2436.0,26330.0
Enroll,782.0,NaN,NaN,NaN,781.424552,928.766979,35.0,242.25,435.5,902.75,6392.0
Top10perc,782.0,NaN,NaN,NaN,27.780051,19.638675,1.0,16.0,23.0,34.0,170.0
Top25perc,782.0,NaN,NaN,NaN,55.767263,19.755277,9.0,41.0,54.0,69.0,100.0
F.Undergrad,782.0,NaN,NaN,NaN,3706.217391,4845.247405,139.0,992.75,1707.5,4118.75,31643.0
P.Undergrad,782.0,NaN,NaN,NaN,855.173913,1518.563712,1.0,95.75,354.0,967.75,21836.0
Outstate,782.0,NaN,NaN,NaN,10379.418159,3809.067321,2340.0,7605.0,9998.0,12478.5,21700.0


### Task 3

*Remove the `'College Name'` feature from the data*

In [5]:
df.drop('College Name', axis = 1, inplace = True)

### Task 4

*Study correlations in the data using correlation matrix for numerical features*

In [6]:
df.corr(numeric_only = True)

,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,...,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Total_Cost,Acceptance_Rate,Faculty_Quality_Index,Student_Success_Index
Apps,1.000000,0.918911,0.836844,0.276677,0.329970,0.807127,0.384348,0.018833,0.129988,-0.015648,...,0.327392,0.342978,0.122954,-0.087517,0.084231,0.137597,0.079123,0.009738,0.383861,0.133491
Accept,0.918911,1.000000,0.911592,0.171500,0.246904,0.874425,0.441961,-0.025967,0.088513,-0.019910,...,0.310435,0.332799,0.188884,-0.161571,0.065895,0.066652,0.038460,-0.099459,0.340585,0.073636
Enroll,0.836844,0.911592,1.000000,0.156306,0.225555,0.964768,0.513802,-0.148399,-0.032476,-0.016393,...,0.293458,0.300398,0.246287,-0.183494,0.042080,-0.010397,-0.057618,-0.055283,0.316395,0.001006
Top10perc,0.276677,0.171500,0.156306,1.000000,0.750353,0.123389,-0.086981,0.467765,0.331261,-0.011935,...,0.407291,0.424047,-0.316605,0.407615,0.153450,0.389786,0.389348,-0.130241,0.655994,0.369132
Top25perc,0.329970,0.246904,0.225555,0.750353,1.000000,0.198509,-0.053457,0.475836,0.311438,0.004478,...,0.479679,0.512966,-0.283678,0.418519,0.148088,0.421610,0.378605,-0.122649,0.660521,0.404898
F.Undergrad,0.807127,0.874425,0.964768,0.123389,0.198509,1.000000,0.571085,-0.202261,-0.067712,-0.007274,...,0.284129,0.290658,0.284292,-0.231520,0.044226,-0.060072,-0.104124,-0.056451,0.299768,-0.049653
P.Undergrad,0.384348,0.441961,0.513802,-0.086981,-0.053457,0.571085,1.000000,-0.231706,-0.051513,0.020240,...,0.120996,0.134583,0.232696,-0.280689,-0.008525,-0.206316,-0.146832,-0.013966,0.096980,-0.169339
Outstate,0.018833,-0.025967,-0.148399,0.467765,0.475836,-0.202261,-0.231706,1.000000,0.608041,0.012139,...,0.315124,0.394769,-0.512826,0.530421,0.159024,0.478709,0.760176,-0.036010,0.440199,0.457792
Room.Board,0.129988,0.088513,-0.032476,0.331261,0.311438,-0.067712,-0.051513,0.608041,1.000000,0.040218,...,0.290536,0.363696,-0.343367,0.271588,0.130159,0.336590,0.574899,-0.061531,0.378668,0.329674
Books,-0.015648,-0.019910,-0.016393,-0.011935,0.004478,-0.007274,0.020240,0.012139,0.040218,1.000000,...,-0.025337,0.024288,-0.019045,-0.066552,0.080544,0.024107,0.129560,-0.007727,-0.003874,0.010318


### Task 5

*One-hot encode all categorical features, separate the data into predictors and target (`'Grad.Rate'`), and do a train-test split of 70-30 using a random state of 42*

In [7]:
X = df.drop(columns = ['Grad.Rate']); y = df['Grad.Rate']
X = pd.get_dummies(X, drop_first = True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30, random_state = 42)

### Task 6

*Scale all predictors to the same [0, 1] scale by training the scaler on the training data and transforming all the predictors using the trained scaler*

In [8]:
scaler = MinMaxScaler(); X_train_s = scaler.fit_transform(X_train); X_test_s = scaler.transform(X_test)

# Unregularised MLR

### Task 7

*Train a linear regression model using all the scaled predictors*

In [9]:
mlr_unreg = LinearRegression(); mlr_unreg.fit(X_train_s, y_train)

LinearRegression()

### Task 8

*Study coefficients of the model and see which ones are declared as high importance*

In [10]:
coef_df = pd.DataFrame({'Coefficient':[mlr_unreg.intercept_] + list(mlr_unreg.coef_)}, index = ['Intercept'] + list(X_train.columns))
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending = False).index); coef_df

,Coefficient
Student_Success_Index,86.335900
Intercept,39.258627
Enroll,-16.220761
F.Undergrad,12.168766
Academic_Standing_Low,-11.570512
Expend,9.860406
Academic_Standing_Medium,-8.385454
Acceptance_Rate,-7.445615
P.Undergrad,-6.520563
Apps,6.165690


### Task 9

*Evaluate performance of the model and study its predictive consistency between training and test sets*

In [11]:
def performance_metrics(lr_model, X0, y0, X1, y1):
    y_pred_train = lr_model.predict(X0)
    y_pred_test = lr_model.predict(X1)
    train_r2 = r2_score(y0, y_pred_train)
    print('R2 score on training data = ', train_r2)
    train_rmse = np.sqrt(mean_squared_error(y0, y_pred_train))
    print('RMSE on training data = ', train_rmse)
    test_rmse = np.sqrt(mean_squared_error(y1, y_pred_test))
    print('RMSE on test data = ', test_rmse)

In [12]:
performance_metrics(mlr_unreg, X_train_s, y_train, X_test_s, y_test)

R2 score on training data =  0.8659904118073796
RMSE on training data =  6.667047638404547
RMSE on test data =  6.257825709194229


# Regularised Models - Lasso

### Task 10

*Train a lasso-regularised linear regression model using the `Lasso()` method with default `alpha` on the training data and study its features and performance (use `random_state = 42` and `max_iter = 500`)*

In [13]:
mlr_lasso = Lasso(random_state = 42, max_iter = 500); mlr_lasso.fit(X_train_s, y_train)

Lasso(max_iter=500, random_state=42)

In [14]:
coef_df = pd.DataFrame({'Coefficient':[mlr_lasso.intercept_] + list(mlr_lasso.coef_)}, index = ['Intercept'] + list(X_train.columns))
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending = False).index); coef_df

,Coefficient
Intercept,63.934570
Student_Success_Index,26.026105
Academic_Standing_Low,-18.588332
Academic_Standing_Medium,-4.676542
Top10perc,0.000000
Top25perc,0.000000
Accept,0.000000
Academic_Standing_High,0.000000
Funding_Type_Religious,0.000000
Funding_Type_Public,-0.000000


In [15]:
performance_metrics(mlr_lasso, X_train_s, y_train, X_test_s, y_test)

R2 score on training data =  0.6814817317784724
RMSE on training data =  10.278572296211234
RMSE on test data =  10.058618937725356


### Task 11

*Use loops to tune over `alpha` values for the optimal value using validation performance (you need to subset your training data again as you cannot use test data for validation purposes, so create a 20% validation data from the training data with `random_state = 42`)*

In [16]:
X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train_s, y_train, test_size = 0.2, random_state = 42)
alphas = np.arange(0, 5.1, 0.1); best_alpha = None; best_mse = np.inf
for alpha in alphas:
    model = Lasso(alpha = alpha, max_iter = 500, random_state = 42)
    model.fit(X_train_sub, y_train_sub)
    y_val_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_val_pred)
    print(f'alpha = {alpha:.6f}, validation MSE = {mse:.4f}')
    if mse < best_mse:
        best_mse = mse
        best_alpha = alpha
print('\nBest alpha:', best_alpha)
print('Best validation MSE:', best_mse)

alpha = 0.000000, validation MSE = 37.5349
alpha = 0.100000, validation MSE = 30.5830
alpha = 0.200000, validation MSE = 28.2217
alpha = 0.300000, validation MSE = 27.6499
alpha = 0.400000, validation MSE = 28.2437
alpha = 0.500000, validation MSE = 29.8153
alpha = 0.600000, validation MSE = 32.3641
alpha = 0.700000, validation MSE = 35.8485
alpha = 0.800000, validation MSE = 40.1988
alpha = 0.900000, validation MSE = 45.5120
alpha = 1.000000, validation MSE = 51.7903
alpha = 1.100000, validation MSE = 59.3319
alpha = 1.200000, validation MSE = 67.8658
alpha = 1.300000, validation MSE = 77.3937
alpha = 1.400000, validation MSE = 87.9181
alpha = 1.500000, validation MSE = 99.4351
alpha = 1.600000, validation MSE = 110.6088
alpha = 1.700000, validation MSE = 115.2963
alpha = 1.800000, validation MSE = 117.4225
alpha = 1.900000, validation MSE = 119.6606
alpha = 2.000000, validation MSE = 122.0098
alpha = 2.100000, validation MSE = 124.4704
alpha = 2.200000, validation MSE = 126.1949
alph

### Task 12

*Train an optimal lasso-regularised linear regression model using the best obtained `alpha` in the `Lasso()` method on the training data and study its features and performance (use same configuration parameters otherwise)*

In [17]:
mlr_lasso = Lasso(alpha = best_alpha, max_iter = 500, random_state = 42)
mlr_lasso.fit(X_train_s, y_train)

Lasso(alpha=0.30000000000000004, max_iter=500, random_state=42)

In [18]:
coef_df = pd.DataFrame({'Coefficient':[mlr_lasso.intercept_] + list(mlr_lasso.coef_)}, index = ['Intercept'] + list(X_train.columns))
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending = False).index); coef_df

,Coefficient
Student_Success_Index,70.705336
Intercept,42.070899
Academic_Standing_Low,-12.310495
Academic_Standing_Medium,-5.813494
Private_Yes,1.005364
Top10perc,0.000000
Top25perc,0.000000
Accept,0.000000
Academic_Standing_High,0.000000
Funding_Type_Religious,0.000000


In [19]:
performance_metrics(mlr_lasso, X_train_s, y_train, X_test_s, y_test)

R2 score on training data =  0.8397781204349127
RMSE on training data =  7.2899832942141805
RMSE on test data =  6.54965593672803


### Task 13

*Train over various values of `alpha` using the `LassoCV` method (set `random_state` as 42 and `max_iter` as 500 and choose a suitable value for `cv`; also note that this method automatically generates `alpha` values to tune over and you don't need to split your training data as the method implements cross-validation) and choose the right alpha (best validation performance is a good indication) and study the features and performance of the optimal model*

In [20]:
mlr_lasso = LassoCV(cv = 5, random_state = 42, max_iter = 500)
mlr_lasso.fit(X_train_s, y_train)
print('Best alpha:', mlr_lasso.alpha_)

Best alpha: 0.028525242376777355


In [21]:
coef_df = pd.DataFrame({'Coefficient':[mlr_lasso.intercept_] + list(mlr_lasso.coef_)}, index = ['Intercept'] + list(X_train.columns))
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending = False).index); coef_df

,Coefficient
Student_Success_Index,85.671880
Intercept,33.905354
Academic_Standing_Low,-11.148547
Academic_Standing_Medium,-7.526495
Expend,6.596908
P.Undergrad,-3.011534
Private_Yes,2.506122
S.F.Ratio,2.175921
Academic_Standing_High,-1.872691
Funding_Type_Religious,1.286480


In [22]:
performance_metrics(mlr_lasso, X_train_s, y_train, X_test_s, y_test)

R2 score on training data =  0.8623572157376853
RMSE on training data =  6.756819952434066
RMSE on test data =  6.082253353325568


# Regularised Models - Ridge

### Task 14

*Train a ridge-regularised linear regression model using the `Ridge()` method with default `alpha` on the training data and study its features and performance (use `random_state = 42` and `max_iter = 500`)*

In [23]:
mlr_ridge = Ridge(random_state = 42, max_iter = 500); mlr_ridge.fit(X_train_s, y_train)

Ridge(max_iter=500, random_state=42)

In [24]:
coef_df = pd.DataFrame({'Coefficient':[mlr_ridge.intercept_] + list(mlr_ridge.coef_)}, index = ['Intercept'] + list(X_train.columns))
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending = False).index); coef_df

,Coefficient
Student_Success_Index,72.007876
Intercept,43.763980
Academic_Standing_Low,-15.526207
Academic_Standing_Medium,-10.294583
Expend,9.441478
S.F.Ratio,5.037204
Enroll,-4.922871
P.Undergrad,-4.539449
Apps,3.904768
Academic_Standing_High,-3.523277


In [25]:
performance_metrics(mlr_ridge, X_train_s, y_train, X_test_s, y_test)

R2 score on training data =  0.8600553784656338
RMSE on training data =  6.813083757026323
RMSE on test data =  6.317260025732456


### Task 15

*Use loops to tune over `alpha` values for the optimal value using validation performance (use the same split of the training data that was used for Lasso regression) and observe feature importances and predictive performance by retraining optimal model on the full training data*

In [26]:
alphas = np.arange(0, 5.1, 0.1); best_alpha = None; best_mse = np.inf
for alpha in alphas:
    model = Ridge(alpha = alpha, max_iter = 500, random_state = 42)
    model.fit(X_train_sub, y_train_sub)
    y_val_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_val_pred)
    print(f'alpha = {alpha:.6f}, validation MSE = {mse:.4f}')
    if mse < best_mse:
        best_mse = mse
        best_alpha = alpha
print('\nBest alpha:', best_alpha)
print('Best validation MSE:', best_mse)

alpha = 0.000000, validation MSE = 37.0907
alpha = 0.100000, validation MSE = 35.2946
alpha = 0.200000, validation MSE = 33.6330
alpha = 0.300000, validation MSE = 32.2814
alpha = 0.400000, validation MSE = 31.1404
alpha = 0.500000, validation MSE = 30.1597
alpha = 0.600000, validation MSE = 29.3087
alpha = 0.700000, validation MSE = 28.5660
alpha = 0.800000, validation MSE = 27.9160
alpha = 0.900000, validation MSE = 27.3462
alpha = 1.000000, validation MSE = 26.8467
alpha = 1.100000, validation MSE = 26.4093
alpha = 1.200000, validation MSE = 26.0270
alpha = 1.300000, validation MSE = 25.6940
alpha = 1.400000, validation MSE = 25.4052
alpha = 1.500000, validation MSE = 25.1561
alpha = 1.600000, validation MSE = 24.9430
alpha = 1.700000, validation MSE = 24.7624
alpha = 1.800000, validation MSE = 24.6114
alpha = 1.900000, validation MSE = 24.4874
alpha = 2.000000, validation MSE = 24.3879
alpha = 2.100000, validation MSE = 24.3108
alpha = 2.200000, validation MSE = 24.2543
alpha = 2.3

In [27]:
mlr_ridge = Ridge(alpha = best_alpha, max_iter = 500, random_state = 42); mlr_ridge.fit(X_train_s, y_train)

Ridge(alpha=2.5, max_iter=500, random_state=42)

In [28]:
coef_df = pd.DataFrame({'Coefficient':[mlr_ridge.intercept_] + list(mlr_ridge.coef_)}, index = ['Intercept'] + list(X_train.columns))
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending = False).index); coef_df

,Coefficient
Student_Success_Index,58.748146
Intercept,48.743538
Academic_Standing_Low,-18.267606
Academic_Standing_Medium,-11.261333
Expend,8.316438
S.F.Ratio,4.071384
Academic_Standing_High,-3.460005
perc.alumni,3.442793
P.Undergrad,-3.373680
Apps,3.147178


In [29]:
performance_metrics(mlr_ridge, X_train_s, y_train, X_test_s, y_test)

R2 score on training data =  0.8444852087266879
RMSE on training data =  7.182100416469823
RMSE on test data =  6.745204722413768


### Task 16

*Train over various values of `alpha` using the `RidgeCV` method (choose a suitable value for `cv` and also note that this method automatically generates `alpha` values to tune over and you don't need to split your training data as the method implements cross-validation) and choose the right alpha (best validation performance is a good indication) and study the features and performance of the optimal model*

In [30]:
mlr_ridge = RidgeCV(cv = 5); mlr_ridge.fit(X_train_s, y_train)
print('Best alpha:', mlr_ridge.alpha_)

Best alpha: 0.1


In [31]:
coef_df = pd.DataFrame({'Coefficient':[mlr_ridge.intercept_] + list(mlr_ridge.coef_)}, index = ['Intercept'] + list(X_train.columns))
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending = False).index); coef_df

,Coefficient
Student_Success_Index,84.583682
Intercept,39.723852
Enroll,-13.086802
Academic_Standing_Low,-12.082903
Expend,9.863820
F.Undergrad,9.535300
Academic_Standing_Medium,-8.638810
Acceptance_Rate,-6.748404
P.Undergrad,-6.155982
Apps,5.807657


In [32]:
performance_metrics(mlr_ridge, X_train_s, y_train, X_test_s, y_test)

R2 score on training data =  0.86587115956178
RMSE on training data =  6.67001340979074
RMSE on test data =  6.232842340235137
